# Explorer l'accidentologie avec Python

[![Ouvrir dans Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bbodin/accidentologie/blob/main/notebooks/accidentologie_demo.ipynb)

Ce notebook montre comment charger, explorer et représenter les données BAAC avec l'API publique `accidentologie`. Le cas d'étude porte sur les **33 communes de la CARA**, de **2020 à 2024 inclus**. Le nom du territoire, ses codes INSEE et la période sont regroupés dans la cellule de configuration afin d'adapter l'analyse à un autre périmètre.

> Les résultats décrivent les accidents et victimes observés. Sans données de trafic, de population exposée ou de kilomètres parcourus, ils ne mesurent pas un risque individuel absolu.

In [ ]:
%pip install --quiet https://github.com/bbodin/accidentologie/archive/refs/heads/main.zip

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import pandas as pd

from accidentologie import (
    ANALYSIS_TABLES,
    Territory,
    download_csv_files,
    get_dataframe,
    get_fatalities_by_user_type,
    get_vehicle_severity_table,
    get_victim_dataframe,
    plot_accident_per_year,
    plot_fatalities_by_user_type,
    plot_mobility_severity,
    plot_monthly_seasonality,
    plot_severity_map,
    plot_severity_map_static,
    plot_victim_age_pyramid,
    plot_vehicle_severity_table,
    plot_weekday_hour_heatmap,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_PATH = DATA_DIR / "reference" / "cara_communes.geojson"
if not REFERENCE_PATH.exists():
    REFERENCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(
        "https://raw.githubusercontent.com/bbodin/accidentologie/main/"
        "data/reference/cara_communes.geojson",
        REFERENCE_PATH,
    )

TERRITORY = Territory.from_geojson(
    REFERENCE_PATH,
    name="Communauté d'agglomération Royan Atlantique (CARA)",
)
TERRITORY_NAME = TERRITORY.name
COMMUNES = TERRITORY.communes
YEARS = range(2020, 2025)
PERIOD_LABEL = f"{min(YEARS)}–{max(YEARS)}"

expected_files = [
    DATA_DIR / str(year) / f"{table}-{year}.csv"
    for year in YEARS
    for table in ANALYSIS_TABLES
]
if not all(path.exists() for path in expected_files):
    print(f"Téléchargement des données BAAC {PERIOD_LABEL}…")
    download_csv_files(
        years=YEARS, data_dir=DATA_DIR, tables=ANALYSIS_TABLES, progress=None
    )
    print("Données prêtes.")

## 1. Charger les accidents

`get_dataframe` retourne une ligne par accident. Les communes sont filtrées avec leurs codes INSEE et les anciens schémas BAAC sont normalisés automatiquement.

In [ ]:
accidents = get_dataframe(
    years=YEARS,
    communes=COMMUNES,
    data_dir=DATA_DIR,
)

summary = pd.DataFrame({
    "indicateur": [
        "Accidents",
        "Accidents géolocalisés",
        "Accidents mortels",
        "Personnes tuées",
        "Véhicules enregistrés",
    ],
    "valeur": [
        len(accidents),
        int(accidents["has_coordinates"].sum()),
        int(accidents["severity"].eq("Mortel").sum()),
        int(accidents["number_killed"].sum()),
        int(accidents["number_vehicles"].sum()),
    ],
})
summary

In [ ]:
accidents.head()

## 2. Construire soi-même un graphique à partir du DataFrame

`get_dataframe` retourne un DataFrame Pandas standard. Cette section agrège les accidents par mois et gravité avec Pandas, puis représente le résultat avec une fonction Matplotlib.

In [ ]:
monthly_by_severity = (
    accidents
    .dropna(subset=["date"])
    .assign(month=lambda frame: frame["date"].dt.to_period("M").dt.to_timestamp())
    .groupby(["month", "severity"])["accident_id"]
    .nunique()
    .unstack(fill_value=0)
)

for severity in ["Léger", "Blessé hospitalisé", "Mortel"]:
    if severity not in monthly_by_severity:
        monthly_by_severity[severity] = 0

monthly_by_severity.tail()

In [ ]:
def plot_custom_monthly_timeline(monthly_dataframe):
    """Trace le nombre mensuel d'accidents par niveau de gravité."""
    columns = ["Léger", "Blessé hospitalisé", "Mortel"]
    colors = ["#FFD800", "#F28C00", "#D00000"]
    ax = monthly_dataframe[columns].plot.area(
        stacked=True,
        color=colors,
        figsize=(12, 5.5),
        alpha=0.85,
        linewidth=0,
    )
    ax.set_title(f"Accidents par mois et gravité — {TERRITORY_NAME}, {PERIOD_LABEL}")
    ax.set_xlabel("Mois")
    ax.set_ylabel("Nombre d'accidents")
    ax.legend(title="Gravité", frameon=False, ncol=3)
    ax.grid(axis="y", alpha=0.25)
    return ax

ax = plot_custom_monthly_timeline(monthly_by_severity)
plt.show()

## 3. Évolution annuelle par gravité

Chaque accident est classé une seule fois, selon la conséquence la plus grave parmi ses usagers.

In [ ]:
ax = plot_accident_per_year(
    accidents,
    title=f"Accidents par année et gravité — {TERRITORY_NAME}, {PERIOD_LABEL}",
)
ax.figure.set_size_inches(11, 5.5)
plt.show()

## 4. Saisonnalité mensuelle

La matrice permet de repérer les mois chargés sans masquer les différences entre années. Une couleur forte signale davantage d'accidents, pas nécessairement davantage de gravité.

In [ ]:
ax = plot_monthly_seasonality(
    accidents,
    title=f"Saisonnalité mensuelle — {TERRITORY_NAME}, {PERIOD_LABEL}",
)
plt.show()

## 5. Répartition par jour de la semaine et heure

Cette heatmap met en évidence les créneaux où les accidents sont les plus nombreux. Elle peut guider une analyse plus fine des trajets domicile-travail, des sorties scolaires ou des périodes de loisirs.

In [ ]:
ax = plot_weekday_hour_heatmap(
    accidents,
    title=f"Répartition des accidents par jour et heure — {TERRITORY_NAME}, {PERIOD_LABEL}",
)
plt.show()

## 6. Véhicules impliqués

Un accident impliquant plusieurs types de véhicules apparaît une fois dans chaque catégorie concernée. Les totaux de ce tableau ne sont donc pas des nombres d'accidents distincts.

In [ ]:
vehicle_table = get_vehicle_severity_table(accidents)
ax = plot_vehicle_severity_table(
    vehicle_table,
    title=f"Accidents par véhicule impliqué et gravité — {TERRITORY_NAME}, {PERIOD_LABEL}",
    max_rows=12,
)
plt.show()

## 7. Charger les usagers et victimes

`get_victim_dataframe` retourne une ligne par usager. Cela permet d'étudier l'âge, le sexe, la catégorie d'usager, le véhicule et le mode de déplacement.

In [ ]:
victims = get_victim_dataframe(
    years=YEARS,
    communes=COMMUNES,
    data_dir=DATA_DIR,
)
victims[["year", "severity", "age", "sex", "user_category", "mobility"]].head()

## 8. Pyramide des âges

La pyramide porte uniquement sur les personnes blessées ou tuées dont l'âge et le sexe sont renseignés. Les hommes sont placés à gauche et les femmes à droite.

In [ ]:
ax = plot_victim_age_pyramid(
    victims,
    title=f"Âge et sexe des victimes blessées ou tuées — {TERRITORY_NAME}, {PERIOD_LABEL}",
)
plt.show()

## 9. Gravité observée par mode de déplacement

Le graphique compare, parmi les victimes blessées ou tuées, la proportion hospitalisée ou décédée. Le seuil de 10 victimes évite d'afficher des pourcentages très instables. **Ce taux n'est pas une probabilité d'accident** : il ne tient pas compte du nombre d'usagers ni des distances parcourues pour chaque mobilité.

In [ ]:
ax = plot_mobility_severity(
    victims,
    min_victims=10,
    title=f"Part hospitalisée ou tuée par mobilité — {TERRITORY_NAME}, {PERIOD_LABEL}",
)
plt.show()

## 10. Personnes tuées par type d'usager

Cette vue compte des personnes, contrairement aux graphiques précédents centrés sur les accidents.

In [ ]:
fatalities = get_fatalities_by_user_type(victims)
ax = plot_fatalities_by_user_type(
    fatalities,
    title=f"Personnes tuées par type d'usager — {TERRITORY_NAME}, {PERIOD_LABEL}",
)
plt.show()

## 11. Carte statique

La carte statique est adaptée à une image ou à un rapport PDF. Étoile rouge : mortel ; losange orange : accident avec blessé hospitalisé ; rond jaune : léger.

In [ ]:
ax = plot_severity_map_static(
    accidents,
    title=f"Accidents géolocalisés — {TERRITORY_NAME}, {PERIOD_LABEL}",
)
ax.figure.set_size_inches(10, 8)
plt.show()

## 12. Carte interactive

Folium produit une carte zoomable avec regroupement des points. La carte est affichée dans le notebook et enregistrée dans `notebooks/outputs/`.

In [ ]:
interactive_map = plot_severity_map(
    accidents,
    output=OUTPUT_DIR / "carte_accidents.html",
)
interactive_map

## Pistes d'analyse supplémentaires

- comparer la saisonnalité avant/après 2020 ;
- séparer semaine et week-end selon le mode de déplacement ;
- calculer les indicateurs par commune, puis les rapporter à la population ;
- intégrer des données de trafic pour passer de volumes observés à des taux d'exposition ;
- étudier séparément les accidents impliquant vélos, motos, piétons ou seniors.